In [1]:
# from transformers import BertModel, BertConfig
from transformers import AutoTokenizer
from datasets import load_dataset
from ray.tune.suggest.hyperopt import HyperOptSearch
from ray.tune.schedulers import ASHAScheduler


In [2]:
dataset = load_dataset("yelp_review_full")

Reusing dataset yelp_review_full (/home/k2141255/.cache/huggingface/datasets/yelp_review_full/yelp_review_full/1.0.0/e8e18e19d7be9e75642fc66b198abadb116f73599ec89a69ba5dd8d1e57ba0bf)


  0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
import transformers
transformers.__version__

'4.19.2'

In [4]:
dataset["train"][100]

{'label': 0,
 'text': 'My expectations for McDonalds are t rarely high. But for one to still fail so spectacularly...that takes something special!\\nThe cashier took my friends\'s order, then promptly ignored me. I had to force myself in front of a cashier who opened his register to wait on the person BEHIND me. I waited over five minutes for a gigantic order that included precisely one kid\'s meal. After watching two people who ordered after me be handed their food, I asked where mine was. The manager started yelling at the cashiers for \\"serving off their orders\\" when they didn\'t have their food. But neither cashier was anywhere near those controls, and the manager was the one serving food to customers and clearing the boards.\\nThe manager was rude when giving me my order. She didn\'t make sure that I had everything ON MY RECEIPT, and never even had the decency to apologize that I felt I was getting poor service.\\nI\'ve eaten at various McDonalds restaurants for over 30 years. 

In [5]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


def tokenize_function(examples):

    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

Loading cached processed dataset at /home/k2141255/.cache/huggingface/datasets/yelp_review_full/yelp_review_full/1.0.0/e8e18e19d7be9e75642fc66b198abadb116f73599ec89a69ba5dd8d1e57ba0bf/cache-17459e8b1dd2fd48.arrow
Loading cached processed dataset at /home/k2141255/.cache/huggingface/datasets/yelp_review_full/yelp_review_full/1.0.0/e8e18e19d7be9e75642fc66b198abadb116f73599ec89a69ba5dd8d1e57ba0bf/cache-919528443e634837.arrow


In [7]:
type(tokenized_datasets)

datasets.dataset_dict.DatasetDict

In [8]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(50))

small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(50))

Loading cached shuffled indices for dataset at /home/k2141255/.cache/huggingface/datasets/yelp_review_full/yelp_review_full/1.0.0/e8e18e19d7be9e75642fc66b198abadb116f73599ec89a69ba5dd8d1e57ba0bf/cache-81ee54f82b61f2f3.arrow
Loading cached shuffled indices for dataset at /home/k2141255/.cache/huggingface/datasets/yelp_review_full/yelp_review_full/1.0.0/e8e18e19d7be9e75642fc66b198abadb116f73599ec89a69ba5dd8d1e57ba0bf/cache-8b13bcabaa2dfb7a.arrow


In [9]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=5)

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.predictions.decoder.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at b

In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(output_dir="test_trainer", evaluation_strategy="epoch")

In [11]:
import numpy as np

from datasets import load_metric

metric = load_metric("accuracy")

In [12]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return metric.compute(predictions=predictions, references=labels)

In [13]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

In [14]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(output_dir="test_trainer", evaluation_strategy="epoch")

In [15]:
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(
        'distilbert-base-uncased', return_dict=True, force_download=True)

In [16]:
trainer = Trainer(

#     model=model,

    args=training_args,

    train_dataset=small_train_dataset,

    eval_dataset=small_eval_dataset,

    compute_metrics=compute_metrics,
    
    model_init=model_init

)

https://huggingface.co/distilbert-base-uncased/resolve/main/config.json not found in cache or force_download set to True, downloading to /home/k2141255/.cache/huggingface/transformers/tmp3f6_41fg


Downloading:   0%|          | 0.00/483 [00:00<?, ?B/s]

storing https://huggingface.co/distilbert-base-uncased/resolve/main/config.json in cache at /home/k2141255/.cache/huggingface/transformers/23454919702d26495337f3da04d1655c7ee010d5ec9d77bdb9e399e00302c0a1.91b885ab15d631bf9cee9dc9d25ece0afd932f2f5130eba28f2055b2220c0333
creating metadata file for /home/k2141255/.cache/huggingface/transformers/23454919702d26495337f3da04d1655c7ee010d5ec9d77bdb9e399e00302c0a1.91b885ab15d631bf9cee9dc9d25ece0afd932f2f5130eba28f2055b2220c0333
loading configuration file https://huggingface.co/distilbert-base-uncased/resolve/main/config.json from cache at /home/k2141255/.cache/huggingface/transformers/23454919702d26495337f3da04d1655c7ee010d5ec9d77bdb9e399e00302c0a1.91b885ab15d631bf9cee9dc9d25ece0afd932f2f5130eba28f2055b2220c0333
Model config DistilBertConfig {
  "_name_or_path": "distilbert-base-uncased",
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "dim": 768,
  "dropout": 0.1,
  "hidden_dim": 3072,

Downloading:   0%|          | 0.00/256M [00:00<?, ?B/s]

storing https://huggingface.co/distilbert-base-uncased/resolve/main/pytorch_model.bin in cache at /home/k2141255/.cache/huggingface/transformers/9c169103d7e5a73936dd2b627e42851bec0831212b677c637033ee4bce9ab5ee.126183e36667471617ae2f0835fab707baa54b731f991507ebbb55ea85adb12a
creating metadata file for /home/k2141255/.cache/huggingface/transformers/9c169103d7e5a73936dd2b627e42851bec0831212b677c637033ee4bce9ab5ee.126183e36667471617ae2f0835fab707baa54b731f991507ebbb55ea85adb12a
loading weights file https://huggingface.co/distilbert-base-uncased/resolve/main/pytorch_model.bin from cache at /home/k2141255/.cache/huggingface/transformers/9c169103d7e5a73936dd2b627e42851bec0831212b677c637033ee4bce9ab5ee.126183e36667471617ae2f0835fab707baa54b731f991507ebbb55ea85adb12a
Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertForSequenceClassification: ['vocab_projector.bias', 'vocab_projector.weight', 'vocab_layer_norm.bias', 'vocab_layer_norm.weig

In [17]:
# trainer.hyperparameter_search(
#     direction="maximize", 
#     backend="ray", 
#     n_trials=10 # number of trials
# )

In [18]:
best_trial = trainer.hyperparameter_search(
    direction="maximize",
    backend="ray",
    # Choose among many libraries:
    # https://docs.ray.io/en/latest/tune/api_docs/suggestion.html
    search_alg=HyperOptSearch(metric="objective", mode="max"),
    # Choose among schedulers:
    # https://docs.ray.io/en/latest/tune/api_docs/schedulers.html
    scheduler=ASHAScheduler(metric="objective", mode="max"))


No `resources_per_trial` arg was passed into `hyperparameter_search`. Setting it to a default value of 1 CPU for each trial.
2022-08-01 10:30:42,147	WARNING tune.py:636 -- Tune detects GPUs, but no trials are using GPUs. To enable trials to use GPUs, set tune.run(resources_per_trial={'gpu': 1}...) which allows Tune to expose 1 GPU to each trial. You can also override `Trainable.default_resource_request` if using the Trainable API.
2022-08-01 10:30:42,310	INFO trial_runner.py:803 -- starting _objective_9cc11e70
2022-08-01 10:30:43,358	INFO trial_runner.py:803 -- starting _objective_9d730810
2022-08-01 10:30:43,366	INFO trial_runner.py:803 -- starting _objective_9d745f76
2022-08-01 10:30:43,375	INFO trial_runner.py:803 -- starting _objective_9d75a444
2022-08-01 10:30:43,383	INFO trial_runner.py:803 -- starting _objective_9d76df58
2022-08-01 10:30:43,425	INFO trial_runner.py:803 -- starting _objective_9d7d5ea0
2022-08-01 10:30:43,433	INFO trial_runner.py:803 -- starting _objective_9d7e9db

== Status ==
Current time: 2022-08-01 10:30:42 (running for 00:00:00.20)
Memory usage on this node: 19.3/62.6 GiB
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 64.000: None | Iter 16.000: None | Iter 4.000: None | Iter 1.000: None
Resources requested: 1.0/16 CPUs, 0/1 GPUs, 0.0/28.9 GiB heap, 0.0/14.45 GiB objects (0.0/1.0 accelerator_type:G)
Result logdir: /home/k2141255/ray_results/_objective_2022-08-01_10-30-42
Number of trials: 1/20 (1 RUNNING)
+---------------------+----------+----------------------+-----------------+--------------------+-------------------------------+---------+
| Trial name          | status   | loc                  |   learning_rate |   num_train_epochs |   per_device_train_batch_size |    seed |
|---------------------+----------+----------------------+-----------------+--------------------+-------------------------------+---------|
| _objective_9cc11e70 | RUNNING  | 10.151.122.12:903333 |     7.32314e-06 |                  3 |                            16

2022-08-01 10:30:43,561	INFO trial_runner.py:803 -- starting _objective_9d9198c0
2022-08-01 10:30:43,580	INFO trial_runner.py:803 -- starting _objective_9d93c0d2
2022-08-01 10:30:43,605	INFO trial_runner.py:803 -- starting _objective_9d9714b2
Downloading: 100%|██████████| 483/483 [00:00<00:00, 521kB/s]
Downloading:   0%|          | 0.00/256M [00:00<?, ?B/s]
Downloading:   1%|          | 1.62M/256M [00:00<00:15, 17.0MB/s]
Downloading:   2%|▏         | 5.26M/256M [00:00<00:08, 29.4MB/s]
Downloading:   4%|▎         | 8.99M/256M [00:00<00:07, 33.3MB/s]
Downloading:   5%|▍         | 12.7M/256M [00:00<00:07, 35.5MB/s]
Downloading:   7%|▋         | 16.6M/256M [00:00<00:06, 37.5MB/s]
Downloading:   8%|▊         | 20.2M/256M [00:00<00:09, 27.0MB/s]
Downloading:   9%|▉         | 23.1M/256M [00:01<00:20, 11.7MB/s]
Downloading:  10%|█         | 25.7M/256M [00:01<00:17, 13.7MB/s]
Downloading: 100%|██████████| 483/483 [00:00<00:00, 999kB/s]
Downloading:  11%|█         | 28.0M/256M [00:01<00:26, 9.11

== Status ==
Current time: 2022-08-01 10:30:48 (running for 00:00:06.50)
Memory usage on this node: 26.6/62.6 GiB
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 64.000: None | Iter 16.000: None | Iter 4.000: None | Iter 1.000: None
Resources requested: 16.0/16 CPUs, 0/1 GPUs, 0.0/28.9 GiB heap, 0.0/14.45 GiB objects (0.0/1.0 accelerator_type:G)
Result logdir: /home/k2141255/ray_results/_objective_2022-08-01_10-30-42
Number of trials: 17/20 (1 PENDING, 16 RUNNING)
+---------------------+----------+----------------------+-----------------+--------------------+-------------------------------+----------+
| Trial name          | status   | loc                  |   learning_rate |   num_train_epochs |   per_device_train_batch_size |     seed |
|---------------------+----------+----------------------+-----------------+--------------------+-------------------------------+----------|
| _objective_9cc11e70 | RUNNING  | 10.151.122.12:903333 |     7.32314e-06 |                  3 |             

Downloading:  24%|██▍       | 60.9M/256M [00:03<00:06, 32.5MB/s]
Downloading:  25%|██▌       | 64.4M/256M [00:03<00:05, 33.7MB/s]
Downloading:  27%|██▋       | 68.1M/256M [00:03<00:05, 35.0MB/s]
Downloading: 100%|██████████| 483/483 [00:00<00:00, 1.20MB/s]
Downloading:  28%|██▊       | 71.6M/256M [00:03<00:05, 35.7MB/s]
Downloading:  29%|██▉       | 75.2M/256M [00:03<00:05, 35.5MB/s]
Downloading:  31%|███       | 79.0M/256M [00:03<00:04, 37.0MB/s]
Downloading:  32%|███▏      | 82.6M/256M [00:03<00:04, 37.0MB/s]
Downloading: 100%|██████████| 483/483 [00:00<00:00, 1.20MB/s]
Downloading:  34%|███▎      | 86.2M/256M [00:04<00:04, 36.8MB/s]
Downloading:  35%|███▌      | 89.8M/256M [00:04<00:06, 28.4MB/s]
Downloading:  36%|███▋      | 92.8M/256M [00:04<00:08, 21.2MB/s]
Downloading: 100%|██████████| 483/483 [00:00<00:00, 1.13MB/s]
Downloading:  38%|███▊      | 96.1M/256M [00:04<00:06, 24.0MB/s]
Downloading:  39%|███▉      | 99.5M/256M [00:04<00:06, 26.6MB/s]
Downloading:  40%|████      | 103M

...

Downloading:  39%|███▉      | 100M/256M [00:04<00:04, 37.7MB/s] 
Downloading:  41%|████      | 104M/256M [00:04<00:04, 37.7MB/s]
Downloading:  42%|████▏     | 107M/256M [00:04<00:04, 37.3MB/s]
Downloading:  43%|████▎     | 111M/256M [00:04<00:04, 37.1MB/s]
Downloading:  45%|████▍     | 114M/256M [00:04<00:03, 37.2MB/s]
Downloading:  46%|████▌     | 118M/256M [00:05<00:03, 37.1MB/s]
2022-08-01 10:31:30,700	INFO trial_runner.py:803 -- starting _objective_b0bbaeb8
Downloading:  48%|████▊     | 122M/256M [00:05<00:03, 37.0MB/s]
Downloading:  49%|████▉     | 125M/256M [00:05<00:03, 36.3MB/s]
Downloading:  50%|█████     | 129M/256M [00:05<00:03, 36.2MB/s]
Downloading:  52%|█████▏    | 132M/256M [00:05<00:08, 15.8MB/s]
Downloading:  53%|█████▎    | 135M/256M [00:06<00:08, 14.1MB/s]
Downloading:  54%|█████▎    | 137M/256M [00:06<00:09, 12.7MB/s]
Downloading:  54%|█████▍    | 139M/256M [00:06<00:10, 11.6MB/s]
Downloading:  55%|█████▍    | 140M/256M [00:06<00:11, 10.8MB/s]
Downloading:  55%|████

== Status ==
Current time: 2022-08-01 10:31:35 (running for 00:00:53.56)
Memory usage on this node: 26.6/62.6 GiB
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 64.000: None | Iter 16.000: None | Iter 4.000: None | Iter 1.000: None
Resources requested: 16.0/16 CPUs, 0/1 GPUs, 0.0/28.9 GiB heap, 0.0/14.45 GiB objects (0.0/1.0 accelerator_type:G)
Result logdir: /home/k2141255/ray_results/_objective_2022-08-01_10-30-42
Number of trials: 20/20 (3 ERROR, 1 PENDING, 16 RUNNING)
+---------------------+----------+----------------------+-----------------+--------------------+-------------------------------+----------+
| Trial name          | status   | loc                  |   learning_rate |   num_train_epochs |   per_device_train_batch_size |     seed |
|---------------------+----------+----------------------+-----------------+--------------------+-------------------------------+----------|
| _objective_9d730810 | RUNNING  | 10.151.122.12:903365 |     4.25489e-05 |                  2 |    

Downloading:  96%|█████████▌| 246M/256M [00:10<00:00, 15.5MB/s]
Downloading:  97%|█████████▋| 248M/256M [00:10<00:00, 14.8MB/s]
Downloading:  98%|█████████▊| 251M/256M [00:11<00:00, 11.9MB/s]
Downloading:  99%|█████████▉| 254M/256M [00:11<00:00, 14.7MB/s]
Downloading: 100%|██████████| 256M/256M [00:11<00:00, 23.2MB/s]
Downloading:   0%|          | 0.00/256M [00:00<?, ?B/s]
Downloading:   1%|          | 1.64M/256M [00:00<00:15, 17.1MB/s]
Downloading:   2%|▏         | 5.05M/256M [00:00<00:09, 28.0MB/s]
(_objective pid=903379) Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertForSequenceClassification: ['vocab_layer_norm.bias', 'vocab_projector.weight', 'vocab_layer_norm.weight', 'vocab_projector.bias', 'vocab_transform.bias', 'vocab_transform.weight']
(_objective pid=903379) - This IS expected if you are initializing DistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architectu

Downloading:  45%|████▍     | 115M/256M [00:04<00:17, 8.48MB/s]
Downloading:  46%|████▌     | 117M/256M [00:05<00:15, 9.62MB/s]
Downloading:  46%|████▋     | 118M/256M [00:05<00:25, 5.66MB/s]
Downloading:  47%|████▋     | 120M/256M [00:05<00:24, 5.72MB/s]
Downloading:  48%|████▊     | 122M/256M [00:06<00:18, 7.44MB/s]
Downloading:  48%|████▊     | 123M/256M [00:06<00:15, 8.80MB/s]
Downloading:  49%|████▉     | 126M/256M [00:06<00:10, 12.4MB/s]
Downloading:  51%|█████     | 131M/256M [00:06<00:07, 18.5MB/s]
Downloading:  52%|█████▏    | 133M/256M [00:06<00:09, 13.9MB/s]
Downloading:  53%|█████▎    | 135M/256M [00:06<00:09, 13.4MB/s]
Downloading:  55%|█████▍    | 140M/256M [00:07<00:05, 20.4MB/s]
Downloading:  56%|█████▋    | 144M/256M [00:07<00:04, 23.7MB/s]
Downloading:  58%|█████▊    | 147M/256M [00:07<00:04, 26.5MB/s]
Downloading:  59%|█████▉    | 151M/256M [00:07<00:03, 28.6MB/s]
Downloading:  60%|██████    | 154M/256M [00:07<00:03, 30.7MB/s]
Downloading:  62%|██████▏   | 158M/256M 

KeyboardInterrupt: 

In [20]:
small_train_dataset['label']

Downloading:  68%|██████▊   | 175M/256M [00:20<00:23, 3.56MB/s]


[4,
 2,
 4,
 0,
 2,
 2,
 3,
 0,
 0,
 4,
 0,
 4,
 3,
 1,
 3,
 0,
 4,
 3,
 4,
 2,
 0,
 4,
 0,
 1,
 0,
 1,
 1,
 2,
 4,
 3,
 3,
 2,
 4,
 3,
 0,
 2,
 2,
 0,
 1,
 2,
 2,
 2,
 0,
 1,
 4,
 1,
 1,
 2,
 0,
 2]

Downloading:  68%|██████▊   | 175M/256M [00:20<00:24, 3.51MB/s]
Downloading:  69%|██████▊   | 175M/256M [00:20<00:24, 3.41MB/s]
Downloading:  69%|██████▊   | 176M/256M [00:20<00:26, 3.17MB/s]
Downloading:  69%|██████▉   | 176M/256M [00:20<00:25, 3.32MB/s]
Downloading:  69%|██████▉   | 176M/256M [00:20<00:24, 3.34MB/s]
Downloading:  69%|██████▉   | 177M/256M [00:21<00:22, 3.64MB/s]
Downloading:  69%|██████▉   | 177M/256M [00:21<00:21, 3.88MB/s]
Downloading:  69%|██████▉   | 178M/256M [00:21<00:20, 4.05MB/s]
Downloading:  70%|██████▉   | 178M/256M [00:21<00:19, 4.16MB/s]
Downloading:  70%|██████▉   | 178M/256M [00:21<00:18, 4.29MB/s]
Downloading:  70%|██████▉   | 179M/256M [00:21<00:18, 4.31MB/s]
Downloading:  70%|███████   | 179M/256M [00:21<00:18, 4.39MB/s]
Downloading:  70%|███████   | 180M/256M [00:21<00:17, 4.47MB/s]
Downloading:  71%|███████   | 180M/256M [00:21<00:16, 4.66MB/s]
Downloading:  71%|███████   | 181M/256M [00:21<00:15, 4.98MB/s]
Downloading:  71%|███████   | 181M/256M 

Downloading:  95%|█████████▍| 242M/256M [00:35<00:05, 2.44MB/s]
Downloading:  95%|█████████▍| 242M/256M [00:35<00:05, 2.44MB/s]
Downloading:  95%|█████████▍| 243M/256M [00:35<00:06, 1.95MB/s]
Downloading:  95%|█████████▌| 243M/256M [00:35<00:08, 1.61MB/s]
Downloading:  95%|█████████▌| 243M/256M [00:36<00:07, 1.85MB/s]
Downloading:  95%|█████████▌| 243M/256M [00:36<00:06, 1.89MB/s]
Downloading:  95%|█████████▌| 244M/256M [00:36<00:06, 1.92MB/s]
Downloading:  95%|█████████▌| 244M/256M [00:36<00:05, 2.19MB/s]
Downloading:  96%|█████████▌| 244M/256M [00:36<00:05, 2.38MB/s]
Downloading:  96%|█████████▌| 244M/256M [00:36<00:05, 2.31MB/s]
Downloading:  96%|█████████▌| 245M/256M [00:36<00:05, 2.17MB/s]
Downloading:  96%|█████████▌| 245M/256M [00:36<00:05, 2.06MB/s]
Downloading:  96%|█████████▌| 245M/256M [00:36<00:04, 2.27MB/s]
Downloading:  96%|█████████▌| 245M/256M [00:37<00:04, 2.37MB/s]
Downloading:  96%|█████████▌| 246M/256M [00:37<00:03, 2.71MB/s]
Downloading:  96%|█████████▋| 246M/256M 

Downloading:  10%|▉         | 24.5M/256M [00:04<01:01, 3.94MB/s]
Downloading:  10%|▉         | 25.0M/256M [00:04<00:56, 4.28MB/s]
Downloading:  10%|▉         | 25.4M/256M [00:04<01:07, 3.58MB/s]
Downloading:  10%|█         | 26.3M/256M [00:04<00:48, 4.99MB/s]
Downloading:  11%|█         | 27.2M/256M [00:04<00:39, 6.05MB/s]
Downloading:  11%|█         | 28.2M/256M [00:05<00:33, 7.14MB/s]
Downloading:  11%|█▏        | 29.3M/256M [00:05<00:28, 8.27MB/s]
Downloading:  12%|█▏        | 30.2M/256M [00:05<00:27, 8.60MB/s]
Downloading:  12%|█▏        | 31.1M/256M [00:05<00:26, 8.90MB/s]
Downloading:  13%|█▎        | 32.2M/256M [00:05<00:24, 9.55MB/s]
Downloading:  13%|█▎        | 33.3M/256M [00:05<00:23, 10.0MB/s]
Downloading:  13%|█▎        | 34.3M/256M [00:05<00:24, 9.49MB/s]
Downloading:  14%|█▍        | 35.2M/256M [00:05<00:25, 9.23MB/s]
Downloading:  14%|█▍        | 36.1M/256M [00:05<00:24, 9.23MB/s]
Downloading:  14%|█▍        | 37.0M/256M [00:05<00:25, 9.16MB/s]
Downloading:  15%|█▍     

Downloading:  56%|█████▌    | 143M/256M [00:16<00:05, 20.9MB/s]
Downloading:  57%|█████▋    | 147M/256M [00:16<00:04, 24.5MB/s]
Downloading:  59%|█████▊    | 150M/256M [00:16<00:04, 27.6MB/s]
Downloading:  60%|██████    | 154M/256M [00:16<00:03, 30.5MB/s]
Downloading:  62%|██████▏   | 158M/256M [00:16<00:03, 32.8MB/s]
Downloading:  63%|██████▎   | 161M/256M [00:16<00:02, 34.6MB/s]
Downloading:  65%|██████▍   | 165M/256M [00:17<00:02, 35.6MB/s]
Downloading:  66%|██████▌   | 168M/256M [00:17<00:02, 31.8MB/s]
Downloading:  68%|██████▊   | 174M/256M [00:17<00:02, 38.5MB/s]
Downloading:  70%|██████▉   | 178M/256M [00:17<00:02, 39.0MB/s]
Downloading:  71%|███████   | 182M/256M [00:17<00:02, 38.7MB/s]
Downloading:  73%|███████▎  | 185M/256M [00:17<00:01, 38.7MB/s]
Downloading:  74%|███████▍  | 189M/256M [00:17<00:01, 38.5MB/s]
Downloading:  75%|███████▌  | 193M/256M [00:17<00:01, 38.6MB/s]
Downloading:  77%|███████▋  | 197M/256M [00:17<00:01, 37.1MB/s]
Downloading:  78%|███████▊  | 200M/256M 

Downloading:  41%|████      | 105M/256M [00:04<00:10, 15.5MB/s]
Downloading:  42%|████▏     | 107M/256M [00:04<00:14, 11.0MB/s]
Downloading:  43%|████▎     | 110M/256M [00:04<00:11, 13.9MB/s]
Downloading:  44%|████▍     | 114M/256M [00:05<00:08, 18.1MB/s]
Downloading:  46%|████▌     | 118M/256M [00:05<00:06, 22.7MB/s]
Downloading:  47%|████▋     | 121M/256M [00:05<00:09, 14.2MB/s]
Downloading:  48%|████▊     | 123M/256M [00:06<00:14, 9.34MB/s]
Downloading:  49%|████▊     | 125M/256M [00:06<00:18, 7.35MB/s]
Downloading:  49%|████▉     | 126M/256M [00:06<00:17, 7.82MB/s]
Downloading:  50%|████▉     | 127M/256M [00:07<00:21, 6.36MB/s]
Downloading:  50%|█████     | 128M/256M [00:07<00:21, 6.34MB/s]
Downloading:  50%|█████     | 129M/256M [00:07<00:19, 6.65MB/s]
Downloading:  51%|█████     | 131M/256M [00:07<00:15, 8.66MB/s]
Downloading:  53%|█████▎    | 134M/256M [00:07<00:09, 13.9MB/s]
Downloading:  54%|█████▍    | 138M/256M [00:07<00:06, 19.3MB/s]
Downloading:  55%|█████▌    | 142M/256M 

(_objective pid=903367) 2022-08-01 10:35:40,985	ERROR function_runner.py:281 -- Runner Thread raised error.
(_objective pid=903367) Traceback (most recent call last):
(_objective pid=903367)   File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/tune/function_runner.py", line 272, in run
(_objective pid=903367)     self._entrypoint()
(_objective pid=903367)   File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/tune/function_runner.py", line 348, in entrypoint
(_objective pid=903367)     return self._trainable_func(
(_objective pid=903367)   File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/util/tracing/tracing_helper.py", line 462, in _resume_span
(_objective pid=903367)     return method(self, *_args, **_kwargs)
(_objective pid=903367)   File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/tune/function_runner.py", line 640, in

2022-08-01 10:35:46,898	ERROR worker.py:92 -- Unhandled error (suppress with 'RAY_IGNORE_UNHANDLED_ERRORS=1'): ray::ImplicitFunc.train() (pid=903367, ip=10.151.122.12, repr=_objective)
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/tune/trainable.py", line 349, in train
    result = self.step()
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/tune/function_runner.py", line 403, in step
    self._report_thread_runner_error(block=True)
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/tune/function_runner.py", line 567, in _report_thread_runner_error
    raise TuneError(
ray.tune.error.TuneError: Trial raised an exception. Traceback:
ray::ImplicitFunc.train() (pid=903367, ip=10.151.122.12, repr=_objective)
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/tune/function_runner.py", line 272, in run
    self._ent